# 19A2B — Efficient Cycle-24 Temporal AIA Staging Plan

## Why this supersedes 19A2A

The first staging-plan notebook queried GCS metadata once per unique object. With tens of thousands of objects, that approach is unnecessarily slow.

This version is **offline and exact**: it reuses the already-recorded `object_bytes` and `object_generation` stored in the completed temporal-manifest evidence.

No model training or Cycle-25 evaluation is performed.

## Outputs

- deduplicated Cycle-24 AIA object inventory;
- exact total payload size from preserved object metadata;
- frame-reuse statistics;
- target-to-object mapping;
- recommended local disk size with 20% headroom.


In [1]:
from pathlib import Path
import gzip, json
from collections import Counter

import pandas as pd

HOME = Path.home()
AIA = HOME / "aia19_temporal_aia_input_corrected"
META = HOME / "aia17_metadata_stage1"
OUT = HOME / "aia19_staging_plan_fast"
OUT.mkdir(parents=True, exist_ok=True)

DEV = AIA / "cycle24_aia_development_manifest.csv.gz"
TEMPORAL = META / "temporal_manifest_v1/reports/20260916T143622218859Z/temporal_sequence_candidates.jsonl.gz"

URI_COLS = [
    "history_uri_tminus288",
    "history_uri_tminus192",
    "history_uri_tminus96",
]

for p in [DEV, TEMPORAL]:
    if not p.exists():
        raise FileNotFoundError(p)

print("DEV:", DEV)
print("TEMPORAL:", TEMPORAL)
print("OUT:", OUT)


DEV: /home/abmoses2000/aia19_temporal_aia_input_corrected/cycle24_aia_development_manifest.csv.gz
TEMPORAL: /home/abmoses2000/aia17_metadata_stage1/temporal_manifest_v1/reports/20260916T143622218859Z/temporal_sequence_candidates.jsonl.gz
OUT: /home/abmoses2000/aia19_staging_plan_fast


## 1. Load the frozen Cycle-24 development manifest

In [2]:
dev = pd.read_csv(DEV)

assert len(dev) == 64725
assert int(dev["label_48h_final"].sum()) == 2094

all_uris = pd.concat([dev[c] for c in URI_COLS], ignore_index=True)
reuse = all_uris.value_counts()

needed = set(reuse.index)
print("Targets:", len(dev))
print("Temporal frame references:", len(all_uris))
print("Unique required NPZ objects:", len(needed))
print("Mean reuse:", len(all_uris) / len(needed))
print("Median reuse:", float(reuse.median()))
print("Max reuse:", int(reuse.max()))


Targets: 64725
Temporal frame references: 194175
Unique required NPZ objects: 70217
Mean reuse: 2.7653559679279947
Median reuse: 3.0
Max reuse: 3


## 2. Recover preserved object metadata from the temporal evidence

In [3]:
objects = {}
rows_scanned = 0
frames_scanned = 0

with gzip.open(TEMPORAL, "rt", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        rows_scanned += 1

        for fr in rec.get("frames", []):
            frames_scanned += 1
            uri = fr.get("object_uri")
            if uri not in needed:
                continue

            if fr.get("object_status") != "EXACT_NONEMPTY_OBJECT":
                raise RuntimeError(f"Required URI is not exact/nonempty: {uri}")

            size = fr.get("object_bytes")
            generation = fr.get("object_generation")

            if not isinstance(size, int) or size <= 0:
                raise RuntimeError(f"Missing preserved object_bytes for {uri}")
            if generation is None:
                raise RuntimeError(f"Missing preserved generation for {uri}")

            record = {
                "object_uri": uri,
                "size_bytes": int(size),
                "generation": str(generation),
                "history_sample_id": fr.get("history_sample_id"),
                "record_utc": fr.get("record_utc"),
            }

            if uri in objects:
                old = objects[uri]
                if old["size_bytes"] != record["size_bytes"] or old["generation"] != record["generation"]:
                    raise RuntimeError(f"Conflicting preserved metadata for {uri}")
            else:
                objects[uri] = record

        if rows_scanned % 25000 == 0:
            print(f"Temporal records scanned: {rows_scanned:,}", flush=True)

missing = needed - set(objects)
unexpected = set(objects) - needed

print("Temporal records scanned:", rows_scanned)
print("Frames scanned:", frames_scanned)
print("Required unique objects recovered:", len(objects))
print("Missing required objects:", len(missing))
print("Unexpected objects:", len(unexpected))

if missing:
    print("First missing:", sorted(missing)[:20])
    raise RuntimeError("Some required Cycle-24 objects lack preserved metadata.")


Temporal records scanned: 25,000


Temporal records scanned: 50,000


Temporal records scanned: 75,000


Temporal records scanned: 100,000


Temporal records scanned: 125,000


Temporal records scanned: 141644
Frames scanned: 424932
Required unique objects recovered: 70217
Missing required objects: 0
Unexpected objects: 0


## 3. Compute exact payload and reuse statistics

In [4]:
obj = pd.DataFrame(objects.values())
obj["reference_count"] = obj["object_uri"].map(reuse).astype(int)

total_bytes = int(obj["size_bytes"].sum())
gib = total_bytes / (1024**3)
gb = total_bytes / 1e9
headroom = gib * 1.20

print("Unique NPZ objects:", len(obj))
print("Total payload bytes:", total_bytes)
print(f"Total payload: {gb:.2f} GB decimal")
print(f"Total payload: {gib:.2f} GiB")
print(f"Recommended free disk (+20%): {headroom:.2f} GiB")

print("\nReuse distribution:")
print(obj["reference_count"].describe().to_string())


Unique NPZ objects: 70217
Total payload bytes: 386895767066
Total payload: 386.90 GB decimal
Total payload: 360.32 GiB
Recommended free disk (+20%): 432.39 GiB

Reuse distribution:


count    70217.000000
mean         2.765356
std          0.582173
min          1.000000
25%          3.000000
50%          3.000000
75%          3.000000
max          3.000000


## 4. Save staging evidence

In [5]:
obj = obj.sort_values(["reference_count", "object_uri"], ascending=[False, True])
obj.to_csv(
    OUT / "cycle24_unique_aia_objects.csv.gz",
    index=False,
    compression="gzip",
)

dev[
    [
        "target_sample_id",
        "label_48h_final",
        "role",
        "region_component_id",
        *URI_COLS,
    ]
].to_csv(
    OUT / "cycle24_temporal_target_to_object_map.csv.gz",
    index=False,
    compression="gzip",
)

summary = {
    "status": "CYCLE24_TEMPORAL_AIA_STAGING_PLAN_READY_FROM_PRESERVED_METADATA_NO_TRAINING",
    "supersedes": "19A2A_Cycle24_Temporal_AIA_staging_plan",
    "targets": int(len(dev)),
    "positives": int(dev["label_48h_final"].sum()),
    "regions": int(dev["region_component_id"].nunique()),
    "temporal_frame_references": int(len(all_uris)),
    "unique_npz_objects": int(len(obj)),
    "mean_reuse_count": float(len(all_uris) / len(obj)),
    "median_reuse_count": float(reuse.median()),
    "max_reuse_count": int(reuse.max()),
    "total_payload_bytes": total_bytes,
    "total_payload_gb_decimal": float(gb),
    "total_payload_gib": float(gib),
    "recommended_free_disk_gib_20pct_headroom": float(headroom),
    "metadata_source": str(TEMPORAL),
    "gcs_per_object_queries_used": False,
    "cycle25_used": False,
    "model_trained": False,
}

(OUT / "staging_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

print(json.dumps(summary, indent=2))


{
  "status": "CYCLE24_TEMPORAL_AIA_STAGING_PLAN_READY_FROM_PRESERVED_METADATA_NO_TRAINING",
  "supersedes": "19A2A_Cycle24_Temporal_AIA_staging_plan",
  "targets": 64725,
  "positives": 2094,
  "regions": 1232,
  "temporal_frame_references": 194175,
  "unique_npz_objects": 70217,
  "mean_reuse_count": 2.7653559679279947,
  "median_reuse_count": 3.0,
  "max_reuse_count": 3,
  "total_payload_bytes": 386895767066,
  "total_payload_gb_decimal": 386.895767066,
  "total_payload_gib": 360.3247618917376,
  "recommended_free_disk_gib_20pct_headroom": 432.3897142700851,
  "metadata_source": "/home/abmoses2000/aia17_metadata_stage1/temporal_manifest_v1/reports/20260916T143622218859Z/temporal_sequence_candidates.jsonl.gz",
  "gcs_per_object_queries_used": false,
  "cycle25_used": false,
  "model_trained": false
}


## 5. Handoff

If the resulting payload fits the planned GPU-VM disk, the next stage can create a deduplicated local cache and train from local storage.

Cycle-25 objects remain untouched until the image model, calibration, and operating threshold are frozen.
